# EDA — Airline Passenger Satisfaction
Fase 2 del proyecto. Análisis exploratorio del dataset elegido en la Fase 1.

## 1. Carga de datos

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency

sns.set_style('whitegrid')

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
df_train = pd.read_csv(f"{path}\\train.csv")
df_test = pd.read_csv(f"{path}\\test.csv")
df = pd.concat([df_train, df_test], ignore_index=True)

print(df.shape)
df.head()

## 2. Limpieza inicial

Antes de empezar el análisis revisamos la estructura del dataframe para detectar cosas raras.

In [ ]:
print(df.info())
print("\nColumnas:", list(df.columns))

Aparece una columna `Unnamed: 0` que no está en el diccionario de variables. La revisamos antes de decidir qué hacer:

In [ ]:
print(df['Unnamed: 0'].head(10))
print(df_train['Unnamed: 0'].equals(df_train.index.to_series()))
print(df_test['Unnamed: 0'].equals(df_test.index.to_series()))

Es el índice original de cada archivo (train y test), y al unirlos ya no significa nada. La quitamos.

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
print(df.shape)

## 3. Valores faltantes

In [ ]:
faltantes = df.isnull().sum()
faltantes_pct = (faltantes / len(df) * 100).round(2)
resumen_faltantes = pd.DataFrame({'faltantes': faltantes, 'porcentaje': faltantes_pct})
print(resumen_faltantes[resumen_faltantes['faltantes'] > 0])

Solo `Arrival Delay in Minutes` tiene datos faltantes: 393 (0.3% del total). Antes de imputar, comparamos media y mediana para decidir con cuál rellenar:

In [ ]:
print("Media:", df['Arrival Delay in Minutes'].mean())
print("Mediana:", df['Arrival Delay in Minutes'].median())
print("% de vuelos sin demora de llegada:", (df['Arrival Delay in Minutes'] == 0).mean() * 100)

La mediana es 0 y más de la mitad de los vuelos (56.3%) no tiene demora. La media (15) está inflada por los pocos vuelos con demoras muy largas. Por eso rellenamos con la mediana y no con la media, así no metemos un valor que no representa a la mayoría.

In [ ]:
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())
print("Faltantes restantes:", df.isnull().sum().sum())

## 4. Outliers

Primero vemos qué columnas numéricas hay en total, para no analizar outliers "a ojo":

In [ ]:
todas_columnas_numericas = df.select_dtypes(include=np.number).columns.tolist()
print(todas_columnas_numericas)

In [ ]:
df[todas_columnas_numericas].describe()

Con el `describe()` de arriba separamos las columnas en tres grupos:

- `id`: es solo un número de pasajero, no sirve para análisis.
- Las columnas de calificación (wifi, comida, comodidad del asiento, etc.) van de 0 a 5. Son escalas de encuesta, no variables continuas — no tiene sentido buscarles "outliers" (un 5 no es un valor raro, es la nota máxima).
- Las que sí son continuas de verdad: `Age`, `Flight Distance`, `Departure Delay in Minutes` y `Arrival Delay in Minutes`. En estas sí vale la pena buscar atípicos.

Nos quedamos con este último grupo para el análisis de outliers.

In [ ]:
columnas_numericas = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), columnas_numericas):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

`Age` no tiene outliers. `Flight Distance` tiene algunos, moderados. Los dos delays tienen muchísimos puntos fuera de la caja — algo esperado, ya que la mayoría de vuelos no tiene demora pero hay una cola de vuelos con demoras grandes.

Ahora lo calculamos con el método IQR para tener un número exacto, no solo la lectura visual:

In [ ]:
def detectar_outliers_iqr(serie):
    Q1, Q3 = serie.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = (serie < lim_inf) | (serie > lim_sup)
    return outliers.sum(), lim_inf, lim_sup

for col in columnas_numericas:
    n_outliers, li, ls = detectar_outliers_iqr(df[col])
    pct = (n_outliers / len(df)) * 100
    print(f"{col}: {n_outliers} outliers ({pct:.2f}%) — rango válido [{li:.1f}, {ls:.1f}]")

Los delays salen con 13-14% de "outliers", un porcentaje demasiado alto para tratarlos como errores. Además, el límite que calcula el IQR (30 minutos) es poco realista para vuelos: una demora de una hora pasa todo el tiempo en la vida real, no es un dato dañado.

Por eso no eliminamos estos outliers. Son vuelos con demoras reales, y de hecho son justo el tipo de caso que puede interesar analizar más adelante (¿la demora afecta la satisfacción?).

## 5. Distribución de las variables numéricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), columnas_numericas):
    sns.histplot(df[col], kde=True, ax=ax, bins=40)
    ax.set_title(col)
plt.tight_layout()
plt.show()

`Age` tiene dos picos (uno cerca de los 25 y otro cerca de los 40), pero es bastante simétrica en general. `Flight Distance` tiene cola hacia la derecha (más vuelos cortos que largos). Los delays son casi todo ceros con una cola larguísima.

Confirmamos esto con el coeficiente de asimetría (skewness):

In [ ]:
for col in columnas_numericas:
    print(f"{col}: skewness = {df[col].skew():.2f}")

`Age` sale casi en 0 (simétrica), `Flight Distance` en 1.11 (sesgo moderado), y los dos delays por encima de 6.5 — un sesgo extremo. Cualquier valor de skewness fuera de ±1 ya se considera sesgado, así que los delays están muy lejos de una distribución normal. Esto confirma por qué el IQR no funcionó bien con ellos: ese método asume datos más simétricos.

Si más adelante se necesitara usar estas variables en un modelo, se podría aplicar una transformación logarítmica para suavizar ese sesgo.

## 6. Variables categóricas

In [ ]:
columnas_categoricas = df.select_dtypes(include=['object', 'str']).columns.tolist()
print(columnas_categoricas)

In [ ]:
for col in columnas_categoricas:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(normalize=True).round(3) * 100)

- `Gender`: casi mitad y mitad.
- `Customer Type`: 82% son clientes leales, solo 18% no leales.
- `Type of Travel`: 69% viaja por trabajo, 31% por motivos personales.
- `Class`: Business y Eco están parejos (48% y 45%), Eco Plus es minoría (7%).
- `satisfaction` (la variable que queremos explicar): 57% no satisfechos, 43% satisfechos. No es un desbalance grave.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), columnas_categoricas):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.show()

## 7. Relación entre variables

### Correlaciones numéricas

Antes de calcular correlaciones, comparamos Pearson contra Spearman en los delays, ya que sabemos que tienen un sesgo fuerte:

In [ ]:
for col1, col2 in [('Departure Delay in Minutes', 'Arrival Delay in Minutes'),
                     ('Departure Delay in Minutes', 'Age'),
                     ('Arrival Delay in Minutes', 'Flight Distance')]:
    pearson = df[col1].corr(df[col2])
    spearman, _ = spearmanr(df[col1], df[col2])
    print(f"{col1} vs {col2}: Pearson={pearson:.3f}, Spearman={spearman:.3f}")

Entre los dos delays, Pearson da 0.96 y Spearman da 0.74 — una diferencia real. Pearson se deja influenciar por los valores extremos y exagera un poco la relación; Spearman, al trabajar con rangos, da un número más realista. En las otras dos comparaciones, donde no hay relación real, da casi lo mismo con cualquiera de los dos métodos.

Con esto en mente, vemos la matriz de correlación completa (incluyendo algunas columnas de calificación):

In [ ]:
columnas_correlacion = columnas_numericas + ['Inflight wifi service', 'Online boarding', 'Seat comfort', 'Inflight entertainment']

corr = df[columnas_correlacion].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

Lo más fuerte es la relación entre los dos delays (0.96, ya la vimos) y entre `Seat comfort` e `Inflight entertainment` (0.61). Los delays no se relacionan con casi nada más.

### Tablas cruzadas con satisfacción

Ahora cruzamos las categóricas contra `satisfaction`, para ver cuáles parecen influir de verdad:

In [ ]:
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class']:
    print(f"\n--- {col} vs satisfaction ---")
    tabla = pd.crosstab(df[col], df['satisfaction'], normalize='index') * 100
    print(tabla.round(1))

`Gender` no marca casi diferencia (57% vs 56%). Las otras tres sí:

- `Customer Type`: los leales están satisfechos el 48% de las veces, los no leales solo el 24%.
- `Type of Travel`: es la más marcada de todas — negocios 58% satisfechos, personal solo 10%.
- `Class`: Business 69% satisfechos, Eco 19%, Eco Plus 25%.

`Type of Travel` y `Class` se mueven muy parecido. Puede que no sean dos efectos separados, sino el mismo fenómeno visto dos veces — lo revisamos:

In [ ]:
pd.crosstab(df['Type of Travel'], df['Class'], normalize='index').round(3) * 100

Confirmado: 66% de los que viajan por negocios van en Business, y 82% de los que viajan por motivos personales van en Eco. Están fuertemente ligadas, así que su efecto sobre la satisfacción probablemente se solapa.

## 8. Hipótesis y validación estadística

A partir de los patrones anteriores, planteamos:

- **H1:** el tipo de viaje influye en la satisfacción (negocios más satisfechos que personal).
- **H2:** la clase influye en la satisfacción (Business más satisfecho que Eco/Eco Plus).
- **H3:** la lealtad del cliente influye en la satisfacción.
- **H4:** `Type of Travel` y `Class` están relacionadas entre sí, así que su efecto sobre la satisfacción puede estar solapado — habría que separarlo con un modelo más adelante.

Probamos H1 con una prueba chi-cuadrado:

H0: no hay relación entre tipo de viaje y satisfacción.
H1: sí hay relación.

In [ ]:
tabla_contingencia = pd.crosstab(df['Type of Travel'], df['satisfaction'])
print(tabla_contingencia)

chi2, p_valor, gl, esperado = chi2_contingency(tabla_contingencia)
print(f"\nChi-cuadrado: {chi2:.2f}")
print(f"Grados de libertad: {gl}")
print(f"Valor p: {p_valor}")

El valor p sale prácticamente 0, mucho menor que 0.05, así que rechazamos H0: sí hay relación entre tipo de viaje y satisfacción, y no es casualidad.

Vale la pena aclarar algo: con una muestra tan grande (129,880 filas), el chi-cuadrado casi siempre va a salir significativo, incluso con diferencias chiquitas. Aquí no es el caso — la diferencia real (58% vs 10%) es enorme, así que el resultado importa tanto estadística como prácticamente.

## 9. Conclusiones

- El tipo de viaje es el factor que más se relaciona con la satisfacción (58% vs 10%), y lo confirmamos con la prueba chi-cuadrado.
- La clase de viaje está pegada al tipo de viaje (negocios → Business, personal → Eco), así que probablemente no son dos efectos independientes sino el mismo patrón visto de dos formas.
- La lealtad del cliente también influye, aunque menos que las dos anteriores.
- Los delays tienen una distribución muy sesgada (skewness ~6.8). El IQR marcó 14% como "outliers", pero decidimos no quitarlos porque son demoras reales, no errores.
- Pearson exagera un poco la correlación entre los delays por su sesgo; Spearman da un número más confiable (0.96 vs 0.74).
- El género no parece influir en la satisfacción.

**Problemas de calidad encontrados:**
- Columna `Unnamed: 0` sin utilidad (se eliminó).
- 393 faltantes en `Arrival Delay in Minutes` (0.3%), rellenados con la mediana.

**Para seguir más adelante:**
- Separar el efecto de `Type of Travel` y `Class` con un modelo de regresión o clasificación, ya que están relacionadas entre sí.
- Probar H2 y H3 también con chi-cuadrado.
- Ver si la duración de la demora (no solo si hay o no) afecta la satisfacción.